In [1]:
import kagglehub
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Sequential,Model
import random

In [2]:
# Download latest version
opath = kagglehub.dataset_download("soumikrakshit/div2k-high-resolution-images")

print("Path to dataset files:", opath)


path=Path(opath + r"/DIV2K_train_HR/DIV2K_train_HR")


list_x=list(path.iterdir())


copy=10
image_i_want_to_prosses=2

Path to dataset files: /kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images


In [3]:
def show(image):
    image=cv2.imread(image)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(image)
    plt.show()

In [4]:
def show_embadding(image):
    plt.imshow(image)
    plt.show()


In [5]:
def make_path_to_embaddings(list_x):
    list=[]
    for path in list_x:
        img=cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        list.append((np.array(img,dtype=np.float32))/255)
    return list

In [6]:
def random_value(x,y):
    x_start = np.random.randint(32, x-32)
    y_start = np.random.randint(32, y-32)
    x_end = np.random.randint(x_start+32, x+1)
    y_end = np.random.randint(y_start+32, y+1)
    return x_start,y_start,x_end,y_end

In [7]:
def crop_images(images):
    list=[]
    for i in images:
        height, width, channels=i.shape
        for j in range(copy):
            x_start, y_start, x_end, y_end=random_value(width, height)
            # print(x_start, y_start, x_end, y_end)
            crop=i[y_start:y_end,x_start:x_end]
            list.append(crop)
    return list

In [8]:
def pixilated_images(image,random_list=[2,4,8,16]):
    x,y,_=image.shape
    facter=np.random.choice(random_list)
    small = cv2.resize(image, (int(y//facter), int(x//facter)), interpolation=cv2.INTER_AREA)
    pixilated = cv2.resize(small, (y, x), interpolation=cv2.INTER_NEAREST)
    return pixilated

In [9]:
def fit_nn(xtrain, ytrain):
    target_h = 1080
    target_w = 1920

    def fix_size(img):
        h, w, c = img.shape

        # Crop if larger (center crop)
        if h > target_h:
            start = (h - target_h) // 2
            img = img[start:start+target_h, :, :]
        if w > target_w:
            start = (w - target_w) // 2
            img = img[:, start:start+target_w, :]

        # Pad if smaller (center pad)
        h, w, c = img.shape
        pad_h = target_h - h
        pad_w = target_w - w

        top = max(0, pad_h) // 2
        bottom = max(0, pad_h) - top
        left = max(0, pad_w) // 2
        right = max(0, pad_w) - left

        if pad_h > 0 or pad_w > 0:
            img = np.pad(
                img,
                ((top, bottom), (left, right), (0, 0)),
                mode="constant",
                constant_values=0
            )

        return img

    xresult = fix_size(xtrain)
    yresult = fix_size(ytrain)

    return xresult, yresult

In [10]:
def loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

In [11]:
def loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))


def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x


def build_unet():
    inputs = layers.Input(shape=(1080,1920, 3))

    # Encoder
    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Bottleneck
    b = conv_block(p2, 64)

    # Decoder
    u2 = layers.UpSampling2D((2, 2))(b)
    u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2, 32)

    u1 = layers.UpSampling2D((2, 2))(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = conv_block(u1, 16)

    # RGB output
    outputs = layers.Conv2D(
        3,
        3,
        padding="same",
        activation="sigmoid",
        dtype="float32"
    )(c6)

    return Model(inputs, outputs)

In [12]:
model = build_unet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=loss
)

I0000 00:00:1789054814.580369      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789054814.583638      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [13]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1080,      │          0 │ -                 │
│ (InputLayer)        │ 1920, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 1080,      │        448 │ input_layer[0][0] │
│                     │ 1920, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1080,      │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 1920, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 1080,      │      2,320 │ batch_normalizat… │
│                     │ 1920, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1080,      │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 1920, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 540, 960,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 540, 960,  │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 540, 960,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 540, 960,  │      9,248 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 540, 960,  │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 270, 480,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 270, 480,  │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 270, 480,  │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 270, 480,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 270, 480,  │        256 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 540, 960,  │          0 │ batch_normalizat… │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 540, 960,  │          0 │ up_sampling2d[0]

 Total params: 119,971 (468.64 KB)

 Trainable params: 119,331 (466.14 KB)

 Non-trainable params: 640 (2.50 KB)

In [14]:
# for i in crop_images(make_path_to_embaddings(list_x=list_x))[0:10]:
#     x,y=fit_nn(pixilated_images(i),i)
#     show_embadding(x)
#     show_embadding(y)

In [15]:
def data_generator():
    paths = list_x.copy()
    random.shuffle(paths)
    for p in paths:
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = np.array(img, dtype=np.float32) / 255
        h, w, _ = img.shape
        for _ in range(copy):
            x_start, y_start, x_end, y_end = random_value(w, h)
            crop = img[y_start:y_end, x_start:x_end]
            xtrain, ytrain = fit_nn(pixilated_images(crop), crop)
            yield xtrain, ytrain

In [ ]:
count=1
loss_values = []
for xtrain,ytrain in data_generator():
    print("=="*40)
    # print(xtrain.shape,ytrain.shape)
    print(">>> count : " + str(count))
    history = model.fit(xtrain[None,...],ytrain[None,...],epochs=2)
    loss_values.extend(history.history["loss"])
    count=count+1
    print("=="*40)
    if (count) % 500 == 0:
        model.save(f"/kaggle/working/model_{count}.keras")
        print("Saved:", count)

>>> count : 1
Epoch 1/2


2026-09-10 15:40:25.836892: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 15:40:25.997525: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 15:40:29.271275: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 15:40:29.452761: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-10 15:40:30.039075: E external/local_xla/xla/stream_

1/1 ━━━━━━━━━━━━━━━━━━━━ 31s 31s/step - loss: 0.2382
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - loss: 0.2353
>>> count : 2
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - loss: 0.2289
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - loss: 0.2273
>>> count : 3
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - loss: 0.2373
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - loss: 0.2365
>>> count : 4
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step - loss: 0.2223
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - loss: 0.2208
>>> count : 5
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - loss: 0.2068
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - loss: 0.2045
>>> count : 6
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - loss: 0.2299
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - loss: 0.2290
>>> count : 7
Epoch 1/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - loss: 0.1739
Epoch 2/2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - loss: 0.1700
>>> count : 8
Epoch 1/2
1/1 ━━━

In [ ]:
def plot_loss(loss_values, window=100):
    import numpy as np
    import matplotlib.pyplot as plt

    smooth_loss = np.convolve(
        loss_values,
        np.ones(window) / window,
        mode="valid"
    )

    plt.figure(figsize=(12, 6))

    plt.plot(
        range(window - 1, len(loss_values)),
        smooth_loss
    )

    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.title(f"Training Loss - Moving Average ({window})")
    plt.grid(True)

    plt.show()

In [ ]:
model = load_model("/kaggle/input/models/chotunaamhaimera/model/keras/default/1/model_8000.keras",compile=False)
model.summary()

In [ ]:
path=Path(opath + r"/DIV2K_valid_HR/DIV2K_valid_HR")
list_x=list(path.iterdir())
images_o=make_path_to_embaddings(list_x)

In [ ]:
No_image=1
images_p=pixilated_images(images_o[No_image],random_list=[16])

In [ ]:
def model_preduct(image):         
    result=model.predict(image[None,...])[0]
    return result

In [ ]:
result=model_preduct(images_p)
# show_embadding(images_p),show_embadding(result),show_embadding(images_o[No_image])

In [ ]:
def show_three_images(images_p, result, original):
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(images_p)
    plt.title("Pixelated Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(result)
    plt.title("Model Output")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_three_images(
    images_p,
    result,
    images_o[No_image]
)